# L5: Adapting the Model Itself

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

So far you've made agents smarter without ever touching the model's weights. First through behavior adaptation (Lesson 2), then through Code Knowledge Graphs (Lessons 3–4).
In this notebook, you'll go one layer deeper and see how to adapt the model itself using **fine-tuning**, a more expensive, but sometimes necessary, tool for agent adaptation.

Here's what you'll do:

- Get an intuition for **LoRA (Low-Rank Adaptation)**: why we freeze the original model weights and train two small matrices (A and B) on top, instead of retraining the whole network
- Explore the trade-off between training too few weights (the new behavior never sticks) and too many (catastrophic forgetting) and why ~1% of weights is often the sweet spot
- Meet the **"super polite" Qwen adapter** — a real LoRA fine-tune trained on ~14,000 synthetic polite Q&A pairs and compare its responses side-by-side against the base model
- Learn how **quantization** (compressing 32-bit weights down to 4 bits) makes training and storing these adapters dramatically cheaper, the adapter ends up around 150MB versus several GB for the base model
- Peek behind the scenes at real training stats: about an hour of training time, ~5% of weights updated, on a 600M-parameter model
- Build a simple **router** that uses pattern matching to decide, query by query, whether to answer with the base model or hand off to a fine-tuned adapter, a preview of how this works in production

By the end, you'll understand not just *how* fine-tuning works, but *when* it's the right tool to reach for compared to the other adaptation strategies you've already built.

In [ ]:
import sys
from pathlib import Path

for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "course_lab").is_dir():
        sys.path.insert(0, str(_p))
        break

from course_lab.sandbox import ensure_ready
ensure_ready()

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> and <code>helper.py</code> files:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>.

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download"</em>.</p>
</div>

In [ ]:
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "course_lab").is_dir():
        sys.path.insert(0, str(_p))
        break

import warnings
warnings.filterwarnings("ignore",
                        message=r".*incompatible torch version.*")
warnings.filterwarnings("ignore",
                        message=r".*incorrect regex pattern.*")

from course_lab import L5_lab, paths
cfg = L5_lab.load_l5_config("weight")
adapters = L5_lab.resolve_weight_adapters()

## A super nice Qwen

Meet the "super polite" LoRA adapter in action — see proof of the personality shift it adds on top of the base model.

In [ ]:
L5_lab.show_politeness_proof()

## The training data

Look at samples from the training data. The polite Q&A pairs used to teach the adapter its tone.

In [ ]:
L5_lab.show_polite_training_samples()

### How QLoRA's 4-bit

See how QLoRA's 4-bit quantization compresses the original 32-bit weights, trading a small amount of precision for a much smaller, cheaper-to-train adapter.

In [ ]:
L5_lab.show_qlora_quantization()

## The cost of finetuning

Review the real training stats: time, percentage of weights updated,and model size. Plot the training loss curve.

In [ ]:
L5_lab.show_training_stats()
L5_lab.plot_loss_curve()

## Test for yourself

Send your own prompt and compare the base model's response side-by-side with the polite adapter's response, live.

In [ ]:
from course_lab import L5_lab

for _q in ['My code in Python doesnt work. What can be the reason?']:
    L5_lab.live_compare_polite(_q); print('-' * 88)

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>Different Run Results:</b> The output generated by AI chat models can vary with each execution due to their dynamic, probabilistic nature. Don't be surprised if your results differ from those shown in the video.</p>

## Plug-and-play model router

See a router that inspects each incoming query and decides whether to answer with the base model or hand off to the fine-tuned adapter.

In [ ]:
L5_lab.show_adapter_router_demo()

## Wrap-up: What You Built

- **Saw the adapter in action**: compared the "super polite" LoRA adapter against the base Qwen model and confirmed the personality shift it adds.

- **Inspected the training data**: looked at samples from the ~14,000 synthetic polite Q&A pairs used to teach the adapter its tone.

- **Understood QLoRA quantization**: saw how compressing weights from 32-bit down to 4-bit makes training and storing the adapter dramatically cheaper, while introducing only a small amount of reconstruction error.

- **Reviewed the real cost**: looked at actual training stats — about an hour of training, ~5% of weights updated on a 600M-parameter model — and plotted the loss curve.

- **Tested it live**: sent a fresh prompt and compared the base model's response against the polite adapter's response side-by-side.

- **Built a router**: implemented a simple pattern-matching router that decides, query by query, whether to answer with the base model or hand off to the fine-tuned adapter, a preview of how this works in production.

- **Connected it to the bigger picture**: understood not just how LoRA fine-tuning works, but when it's the right (if more expensive) tool to reach for compared to behavior adaptation and Code Knowledge Graphs.